In [ ]:
import numpy as np
import mne
import matplotlib.pyplot as plt

In [ ]:
class utils:

    @staticmethod
    def plot_wavelets(wavelets, fs):
        if not isinstance(wavelets, list):
            wavelets = [wavelets]
        for wavelet in wavelets:
            n_samples = len(wavelet)
            t = np.arange(n_samples) / fs - (n_samples - 1) / (2 * fs )
            plt.plot(t, wavelet.real)
            plt.xlabel("Time [s]")
            plt.ylabel("Amplitude [a.u.]")

## Constructing Morlet Wavelts

In [ ]:
fs = 125
freq = 5
n_cycles = 7

wavelet = mne.time_frequency.morlet(sfreq=fs, freqs=freq, n_cycles=n_cycles)
utils.plot_wavelets(wavelet, fs)


## Applying Wavelet Transformation

Run the cell below to simulate 2 EEG channels with a transient 20 Hz oscillation.

In [ ]:
sfreq=512
ch_names = ["CH01", "CH02"]
info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")

n_times = 550
n_epochs = 40
seed = 42
rng = np.random.RandomState(seed)
data = rng.randn(len(ch_names), n_times * n_epochs + 200)  # buffer

t = np.arange(n_times, dtype=np.float64) / sfreq
signal = np.sin(np.pi * 2.0 * 20.0 * t)
signal[np.logical_or(t < 0.4, t > 0.6)] = 0.0  # hard windowing
on_time = np.logical_and(t >= 0.4, t <= 0.6)
signal[on_time] *= np.hanning(on_time.sum())  # ramping
data[:, 100:-100] += np.tile(signal, n_epochs)  # add signal

raw = mne.io.RawArray(data, info)
events = np.zeros((n_epochs, 3), dtype=int)
events[:, 0] = np.arange(n_epochs) * n_times
epochs = mne.epochs.Epochs(
    raw,
    events,
    tmin=0,
    tmax=n_times / sfreq,
    baseline=None,
)

epochs.average().plot();

In [ ]:
power = epochs.compute_tfr(method="morlet", freqs=[20], n_cycles=7, average=True)
plt.plot(epochs.times, power.get_data()[0, 0, :])
plt.xlabel("Time [s]")
plt.ylabel("Amplitude [a.u.]")

In [ ]:
power = epochs.compute_tfr(method="morlet", freqs=[20], n_cycles=3, average=True)
plt.plot(epochs.times, power.get_data()[0, 0, :])
plt.xlabel("Time [s]")
plt.ylabel("Amplitude [a.u.]")

In [ ]:
power = epochs.compute_tfr(method="morlet", freqs=[20], n_cycles=1, average=True)
plt.plot(epochs.times, power.get_data()[0, 0, :])
plt.xlabel("Time [s]")
plt.ylabel("Amplitude [a.u.]")

In [ ]:
power = epochs.compute_tfr(method="morlet", freqs=[15], n_cycles=8, average=True)
plt.plot(epochs.times, power.get_data()[0, 0, :])
plt.xlabel("Time [s]")
plt.ylabel("Amplitude [a.u.]")

In [ ]:
power = epochs.compute_tfr(method="morlet", freqs=[15], n_cycles=12, average=True)
plt.plot(epochs.times, power.get_data()[0, 0, :])
plt.xlabel("Time [s]")
plt.ylabel("Amplitude [a.u.]")

## Two-Dimensional Time-Frequency Plots

In [ ]:
freqs = np.arange(5.0, 40.0, 3.0)  # Define full frequency range
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharey=True, layout="constrained")
all_n_cycles = [1, 3, freqs / 2.0]
for n_cycles, ax in zip(all_n_cycles, axs):
    power = epochs.compute_tfr(
        method="morlet", freqs=freqs, n_cycles=n_cycles, return_itc=False, average=True
    )
    power.plot(
        [0],
        baseline=(0.0, 0.1),
        mode="mean",
        axes=ax,
        vlim=(-3, 3),
        show=False,
    )
    n_cycles = "scaled by freqs" if not isinstance(n_cycles, int) else n_cycles
    ax.set_title(f"Sim: Using Morlet wavelet, n_cycles = {n_cycles}")